# CIDA-Rescue — рой, который спасает людей вместе

Здание-лабиринт. Тяжёлых пострадавших несут только вместе, завалы разбирают только вместе, пожар обходят,
двери внезапно заваливает. Каждый дрон видит лишь окно вокруг себя.

**Конвейер:**
1. базовый рой с общей картой учится с нуля (летать, искать, носить);
2. от его лучшей версии три одинаковых дообучения: **без связи** (просто дольше), **обычное внимание**, **CIDA** —
   одинаковый бюджет, разница только в связи, поэтому сравнение честное;
3. таблица по уровням сложности и HTML-демо: одна сцена — три роя рядом.

Runtime → **A100**. Всё сохраняется на Google Drive; после отключения просто запустите ноутбук заново —
готовые этапы пропустятся.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys, json
if not os.path.exists("/content/CIDA-plugin"):
    !git clone https://github.com/Kairatzh/CIDA-plugin.git /content/CIDA-plugin
%cd /content/CIDA-plugin
!git pull --ff-only
!git log --oneline -1
from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/cida_rescue_runs"
os.makedirs(RUNS, exist_ok=True)

import torch
gpu = torch.cuda.get_device_properties(0)
mem_gb = gpu.total_memory / 2**30
# число миров под память карты: A100 40/80 ГБ — 16384, L4 24 ГБ — 8192, T4 15 ГБ — 4096
N_ENVS = 16384 if mem_gb > 30 else 8192 if mem_gb > 20 else 4096
print(f"GPU: {gpu.name}, {mem_gb:.0f} ГБ → {N_ENVS} миров")
if "A100" not in gpu.name and "H100" not in gpu.name:
    print("ВНИМАНИЕ: это не A100 — обучение будет в разы медленнее. Runtime → Change runtime type → A100.")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORLD = f"--world rescue --n-drones 6 --min-drones 3 --n-victims 3 --max-steps 300 --share-coverage --n-envs {N_ENVS}"

def run(name, flags):
    if os.path.exists(f"{RUNS}/{name}/result.json"):
        print("готово, пропускаю:", name); return
    cmd = f"{sys.executable} -u -m cida_swarm.train {WORLD} {flags} --name {name} --out {RUNS}/{name}"
    print(">>>", cmd, flush=True)
    get_ipython().system(cmd)
    if not os.path.exists(f"{RUNS}/{name}/result.json"):
        raise RuntimeError(f"{name} не завершился — смотрите лог выше")

## 1. Базовый рой (~1 час, точнее — по скорости в логе)

Учится с нуля на смеси миров всех уровней сложности. Лучшая версия сохраняется в `model_best.pt`.

**Стоп-сигнал, чтобы не жечь кредиты:** случайный рой выносит около 1% людей (`rescued_frac 0.01`).
Если к 150-й итерации (`[eval] it 150`) `rescued_frac` в строке `[eval]` всё ещё ниже 0.03 — остановите ячейку (■)
и пришлите лог: значит, рой не ухватил переноску, и это надо чинить, а не ждать.
Время всего этапа = 3e9 / (agent-steps/s из лога) секунд.

In [ ]:
run("base", "--total-steps 3e9 --eval-every 25 --seed 0")

## 2. Три дообучения с одинаковым бюджетом (~25 минут каждое)

Все стартуют с лучшей версии базового роя. Меньше случайности (`ent-coef`) и маленький шаг обучения —
чтобы не испортить то, что рой уже умеет.

In [ ]:
FT = f"--init-from {RUNS}/base/model_best.pt --lr 1e-4 --ent-coef 0.001 --critic-warmup 3 --total-steps 1.5e9 --eval-every 25 --seed 0"
run("ft_none", f"{FT} --comm none")
run("ft_attn", f"{FT} --comm attn")
run("ft_cida", f"{FT} --comm cida")

## 3. Итоги

In [ ]:
names = {"base": "базовый (общая карта)", "ft_none": "+ дообучение без связи",
         "ft_attn": "+ обычное внимание", "ft_cida": "+ CIDA"}
print("| Рой | Вынесено людей | Все вынесены | Шагов | Дронов потеряно | Ур.1 | Ур.2 | Ур.3 | Ур.4 |")
print("|---|---|---|---|---|---|---|---|---|")
for key, label in names.items():
    path = f"{RUNS}/{key}/result.json"
    if not os.path.exists(path): continue
    b = json.load(open(path))["best"]
    lv = [f"{b['by_level'].get(str(l), {}).get('rescued_frac', float('nan')):.1%}" for l in range(1, 5)]
    print(f"| {label} | {b['rescued_frac']:.1%} | {b['all_rescued']:.1%} | {b['length']:.0f} | "
          f"{b['drones_lost']:.1%} | " + " | ".join(lv) + " |")

## 4. Демо

Одна сцена — три роя рядом. Файл `rescue_demo.html` сохраняется на Drive: скачайте и откройте в браузере
(работает без интернета, подходит для проектора).

In [ ]:
out = f"{RUNS}/rescue_demo.html"
!{sys.executable} -m cida_swarm.demo {RUNS}/ft_none/model_best.pt {RUNS}/ft_attn/model_best.pt {RUNS}/ft_cida/model_best.pt     --labels "без связи" "обычное внимание" "CIDA" --scenes 6 --difficulty 2 3 4 --out {out}
from IPython.display import HTML, display
display(HTML(open(out, encoding="utf-8").read()))